<a href="https://colab.research.google.com/github/sanaisrail/urdu-ocr-codesaviours-si26-Sana/blob/main/Week4_Urdu_OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install -q transformers==4.46.3 sentencepiece datasets accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [7]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dataset (1).csv")

print(df.head())
print(df.columns)
print("Total Samples:", len(df))

                                      image  \
0   /content/drive/MyDrive/data/other/1.png   
1  /content/drive/MyDrive/data/other/20.png   
2   /content/drive/MyDrive/data/other/5.png   
3  /content/drive/MyDrive/data/other/35.png   
4  /content/drive/MyDrive/data/other/34.png   

                                                text  
0  ایران کے گلستان صوبے میں واقع آقتکہ خان ریلوے ...  
1  روان سال فروری میں، ایران کے خلاف امریکہ اور ا...  
2  اس پل کی اہمیت کو دو اہم بین الاقوامی راستوں ک...  
3  ایرانی ریلوے حکام کے مطابق، گذشتہ سال کم از کم...  
4         پٹرول سے سیرامکس تک تجارت کا ایک اہم راستہ  
Index(['image', 'text'], dtype='object')
Total Samples: 200


In [8]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = model.to(device)

print("✅ Model loaded successfully")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded successfully


In [9]:
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Decoder Start Token:", model.config.decoder_start_token_id)
print("Pad Token:", model.config.pad_token_id)
print("EOS Token:", model.config.eos_token_id)

Decoder Start Token: 0
Pad Token: 1
EOS Token: 2


In [10]:
from PIL import Image
from torch.utils.data import Dataset

class OCRDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # CSV already contains full path
        image_path = row["image"]

        image = Image.open(image_path).convert("RGB")

        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [16]:
from torch.utils.data import DataLoader, random_split

# Dataset
dataset = OCRDataset(df, processor)

# Train/Test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

# Loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Train samples: 160
Test samples: 40


In [38]:
dataset = OCRDataset(
    df,
    processor
)

print("Dataset Loaded:", len(dataset))

Dataset Loaded: 200


In [19]:
dataset = OCRDataset(df, processor)

train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

In [20]:
from transformers import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("Optimizer Ready")

Optimizer Ready


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [21]:
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Decoder Start:", model.config.decoder_start_token_id)
print("Pad:", model.config.pad_token_id)
print("EOS:", model.config.eos_token_id)

Decoder Start: 0
Pad: 1
EOS: 2


In [40]:
import torch
from transformers import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.train()

optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 30

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    total_loss = 0

    for batch in train_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Loss:", total_loss / len(train_loader))


Epoch 1/30
Loss: 3.461907925605774

Epoch 2/30
Loss: 3.4779323291778566

Epoch 3/30
Loss: 3.4529004001617434

Epoch 4/30
Loss: 3.4053202772140505

Epoch 5/30
Loss: 3.409335515499115

Epoch 6/30
Loss: 3.386127290725708

Epoch 7/30
Loss: 3.3649222564697268

Epoch 8/30
Loss: 3.31334942817688

Epoch 9/30
Loss: 3.3588242602348326

Epoch 10/30
Loss: 3.3166852831840514

Epoch 11/30
Loss: 3.2766199779510496

Epoch 12/30
Loss: 3.263721191883087

Epoch 13/30
Loss: 3.236534068584442

Epoch 14/30
Loss: 3.231765418052673

Epoch 15/30
Loss: 3.2364390969276426

Epoch 16/30
Loss: 3.212667820453644

Epoch 17/30
Loss: 3.2028241777420043

Epoch 18/30
Loss: 3.2020414519309996

Epoch 19/30
Loss: 3.1980333042144777

Epoch 20/30
Loss: 3.142600169181824

Epoch 21/30
Loss: 3.166461088657379

Epoch 22/30
Loss: 3.1560419821739196

Epoch 23/30
Loss: 3.1388850593566895

Epoch 24/30
Loss: 3.154122669696808

Epoch 25/30
Loss: 3.1531442904472353

Epoch 26/30
Loss: 3.1285210466384887

Epoch 27/30
Loss: 3.129618947505

In [23]:
type(batch)

dict

In [ ]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=128,
        num_beams=4,
        early_stopping=True
    )


In [41]:
import torch

model.eval()

correct = 0
total = 0

predictions_list = []
actuals_list = []

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

        predictions = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        labels = labels.clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        actuals = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(predictions, actuals):

            predictions_list.append(pred)
            actuals_list.append(actual)

            if pred.strip() == actual.strip():
                correct += 1

            total += 1

accuracy = (correct / total) * 100

print(f"\nAccuracy: {accuracy:.2f}%")


Accuracy: 0.00%


In [25]:
batch = next(iter(test_loader))

print(type(batch))

if isinstance(batch, dict):
    print(batch.keys())
else:
    print(batch)

<class 'dict'>
dict_keys(['pixel_values', 'labels'])


In [26]:
from PIL import Image

model.eval()

image = Image.open("/content/drive/MyDrive/data/other/1.png").convert("RGB")

pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

with torch.no_grad():
    generated_ids = model.generate(pixel_values)

prediction = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("Prediction:", prediction)

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Prediction: اااااااااااااااااا


In [ ]:
print(model.config.decoder_start_token_id)
print(model.config.pad_token_id)
print(model.config.eos_token_id)

0
1
2


In [36]:
batch = next(iter(train_loader))

print(batch["pixel_values"].shape)
print(batch["labels"].shape)

print(
    processor.tokenizer.decode(
        batch["labels"][0].masked_fill(
            batch["labels"][0] == -100,
            processor.tokenizer.pad_token_id
        ),
        skip_special_tokens=True
    )
)

torch.Size([2, 3, 384, 384])
torch.Size([2, 128])
ایک اور جگہ اللہ تعالیٰ کا فرمان ہے کہ
